In [5]:
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline


In [6]:
def load_documents(file):
    loader = PyPDFLoader(file)
    return loader.load()


def split_documents(documents):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=150
    )
    return splitter.split_documents(documents)


def create_vectorstore(chunks):
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    return FAISS.from_documents(chunks, embeddings)

In [7]:
def build_rag(vectorstore):

    pipe = pipeline(
    task="text2text-generation",
    model="google/flan-t5-base",   
    max_new_tokens=60,
    temperature=0.0,
    do_sample=False
)
  

    llm = HuggingFacePipeline(pipeline=pipe)

    prompt = ChatPromptTemplate.from_template(
        """Answer ONLY from the given context.
If answer is not in context, say: "Not found in document".

Give one short sentence only.

Context:
{context}

Question:
{input}

Answer:"""
    )

    retriever = vectorstore.as_retriever(
        search_kwargs={"k": 3}
    )

    document_chain = create_stuff_documents_chain(llm, prompt)
    rag_chain = create_retrieval_chain(retriever, document_chain)

    return rag_chain, llm

def create_agent(llm, rag_chain):

    def clean_output(text):
        text = text.strip()

        if "Answer:" in text:
            text = text.split("Answer:")[-1].strip()

        text = text.split("\n")[0]
        return text

    def qa_tool(query):
        result = rag_chain.invoke({"input": query})
        return clean_output(result["answer"])

    def summarize_tool(text):
        result = llm.invoke("Summarize in one short sentence: " + text)
        return clean_output(result)

    def explain_tool(text):
        result = llm.invoke("Explain in two short sentences: " + text)
        return clean_output(result)

    def agent(query):
        q = query.lower()

        if "summarize" in q:
            return summarize_tool(query)
        elif "explain" in q:
            return explain_tool(query)
        else:
            return qa_tool(query)

    return agent



In [8]:
def main():
    print("📘 AI Knowledge Assistant\n")

    file_path = "ai_notes.pdf"

    print("Processing document...\n")

    docs = load_documents(file_path)
    chunks = split_documents(docs)
    vectorstore = create_vectorstore(chunks)

    rag_chain, llm = build_rag(vectorstore)
    agent = create_agent(llm, rag_chain)

    print("✅ Ready! Ask questions (type 'exit' to quit)\n")

    while True:
        query = input("You: ")
        print(query)

        if query.lower() == "exit":
            print("Goodbye 👋")
            break

        try:
            response = agent(query)
            print("\nAI:", response, "\n")
        except Exception as e:
            print("\n⚠️ Error:", str(e), "\n")

if __name__ == "__main__":
    main()

📘 AI Knowledge Assistant

Processing document...



Device set to use mps:0


✅ Ready! Ask questions (type 'exit' to quit)

What Is Artificial Intelligence

AI: The simulation of human intelligence in machines 

quit

AI: Not found in document 

exit
Goodbye 👋
